# Sistema Esperto per Diagnosi Medica con IBM LNN

Questo notebook dimostra come utilizzare **IBM Logical Neural Networks (LNN)** per creare un sistema di diagnosi medica basato su regole logiche.

## Cosa Imparerai

- Definire predicati e regole logiche con LNN
- Gestire incertezza con bounds (lower, upper)
- Eseguire inferenza bidirezionale (modus ponens + tollens)
- Creare un sistema esperto interpretabile

## Scenario

Creiamo un sistema che inferisce malattie (Influenza, Raffreddore, Covid, Bronchite) da sintomi osservati, gestendo l'incertezza nelle osservazioni.

---

**Nota sull'architettura**: Questo notebook segue le best practice DRY (Don't Repeat Yourself). La logica del sistema è definita nel file `medical_diagnosis.py` e importata qui per la demo interattiva.

## Setup Ambiente

Installiamo le dipendenze e prepariamo il modulo per l'import.

In [27]:
# Installazione dipendenze
!pip install -q git+https://github.com/IBM/LNN.git
!pip install -q torch>=2.0.0 numpy>=1.24.0

# ============================================
# Auto-download del modulo per Google Colab
# ============================================
# Su Colab, scarichiamo il file .py da GitHub.
# In locale, il file è già presente.

try:
    import google.colab
    IN_COLAB = True
    print("🌐 Esecuzione su Google Colab")
except ImportError:
    IN_COLAB = False
    print("💻 Esecuzione locale")

if IN_COLAB:
    # Scarica il modulo .py da GitHub
    import urllib.request
    url = "https://raw.githubusercontent.com/gianlucamazza/neuro_llm/main/examples/01_medical_expert/medical_diagnosis.py"

    print("📥 Download modulo medical_diagnosis.py...")
    urllib.request.urlretrieve(url, "medical_diagnosis.py")
    print("✓ Modulo scaricato correttamente")
else:
    print("✓ Usando file locale medical_diagnosis.py")

print("\n" + "="*50)
print("Setup completato!")
print("="*50)

  Preparing metadata (setup.py) ... done
🌐 Esecuzione su Google Colab
📥 Download modulo medical_diagnosis.py...
✓ Modulo scaricato correttamente

Setup completato!


## Import Moduli

Importiamo la classe `MedicalDiagnosisSystem` dal modulo.

In [28]:
from medical_diagnosis import MedicalDiagnosisSystem
from lnn import Fact

print("✓ MedicalDiagnosisSystem caricato correttamente")
print("✓ Pronto per creare diagnosi!")

✓ MedicalDiagnosisSystem caricato correttamente
✓ Pronto per creare diagnosi!


## Architettura del Sistema

Il `MedicalDiagnosisSystem` implementa:

### Predicati (Neuroni Logici)
- **Sintomi**: Ha_Febbre, Ha_Tosse, Ha_Mal_Di_Gola, Ha_Dolori_Muscolari, Ha_Congestione, Ha_Respiro_Corto
- **Diagnosi**: Influenza, Raffreddore, Covid, Bronchite

### Regole Logiche
1. **Influenza**: Febbre ∧ Dolori_Muscolari → Influenza
2. **Raffreddore**: Tosse ∧ Mal_Di_Gola ∧ ¬Febbre → Raffreddore
3. **Covid**: Febbre ∧ Tosse ∧ (Dolori ∨ Congestione) → Covid
4. **Bronchite**: Tosse ∧ Respiro_Corto ∧ Dolori → Bronchite

## Creazione del Sistema

Istanziamo il sistema esperto.

In [29]:
print("="*60)
print("SISTEMA ESPERTO DI DIAGNOSI MEDICA con LNN")
print("="*60)

# Crea il sistema
system = MedicalDiagnosisSystem()

print("\n✓ Sistema creato con successo!")
print("✓ Regole caricate: Influenza, Raffreddore, Covid, Bronchite")

SISTEMA ESPERTO DI DIAGNOSI MEDICA con LNN

✓ Sistema creato con successo!
✓ Regole caricate: Influenza, Raffreddore, Covid, Bronchite


## Caso 1: Mario - Sintomi Influenzali Chiari

Paziente con **febbre alta** e **dolori muscolari** evidenti. La tosse è presente ma con incertezza.

Sintomi:
- Febbre: ✓ (certa)
- Dolori muscolari: ✓ (certi)
- Tosse: ~ (moderata, bounds [0.6, 0.8])
- Mal di gola: ✗ (assente)

In [30]:
print("[Caso 1] Paziente Mario - Sintomi influenzali")

system.add_patient('Mario', {
    'febbre': Fact.TRUE,
    'dolori_muscolari': Fact.TRUE,
    'tosse': (0.6, 0.8),  # Tosse moderata (incertezza)
    'mal_di_gola': Fact.FALSE,
})

print("✓ Paziente Mario aggiunto al sistema")

[Caso 1] Paziente Mario - Sintomi influenzali
✓ Paziente Mario aggiunto al sistema


## Caso 2: Anna - Raffreddore Classico

Sintomi tipici di raffreddore: **tosse** e **mal di gola** senza febbre.

Sintomi:
- Febbre: ✗ (assente - importante!)
- Tosse: ✓ (presente)
- Mal di gola: ✓ (presente)
- Congestione: ✓ (presente)

In [31]:
print("[Caso 2] Paziente Anna - Raffreddore")

system.add_patient('Anna', {
    'febbre': Fact.FALSE,
    'tosse': Fact.TRUE,
    'mal_di_gola': Fact.TRUE,
    'congestione': Fact.TRUE,
})

print("✓ Paziente Anna aggiunto al sistema")

[Caso 2] Paziente Anna - Raffreddore
✓ Paziente Anna aggiunto al sistema


## Caso 3: Luigi - Possibile Covid

Sintomi che si sovrappongono a Covid: **febbre**, **tosse**, **dolori** e **congestione**.

Sintomi:
- Febbre: ✓ (presente)
- Tosse: ✓ (presente)
- Dolori muscolari: ~ (probabili, bounds [0.7, 0.9])
- Congestione: ✓ (presente)

In [32]:
print("[Caso 3] Paziente Luigi - Possibile Covid")

system.add_patient('Luigi', {
    'febbre': Fact.TRUE,
    'tosse': Fact.TRUE,
    'dolori_muscolari': (0.7, 0.9),
    'congestione': Fact.TRUE,
})

print("✓ Paziente Luigi aggiunto al sistema")

[Caso 3] Paziente Luigi - Possibile Covid
✓ Paziente Luigi aggiunto al sistema


## Caso 4: Sara - Bronchite

**Tosse persistente** con **respiro corto** e dolori, senza febbre.

Sintomi:
- Tosse: ✓ (presente)
- Respiro corto: ✓ (presente)
- Dolori muscolari: ~ (moderati, bounds [0.6, 0.8])
- Febbre: ✗ (assente)

In [33]:
print("[Caso 4] Paziente Sara - Bronchite")

system.add_patient('Sara', {
    'tosse': Fact.TRUE,
    'respiro_corto': Fact.TRUE,
    'dolori_muscolari': (0.6, 0.8),
    'febbre': Fact.FALSE,
})

print("✓ Paziente Sara aggiunto al sistema")

[Caso 4] Paziente Sara - Bronchite
✓ Paziente Sara aggiunto al sistema


## Esecuzione Inferenza

LNN esegue **inferenza bidirezionale** applicando le regole logiche ai sintomi osservati.

Inferenza bidirezionale significa:
- **Modus ponens**: A ∧ B → C (se A e B sono veri, deduce C)
- **Modus tollens**: ¬C → ¬(A ∧ B) (se C è falso, deduce che A o B sono falsi)

In [34]:
print("\n" + "="*60)
print("Esecuzione inferenza LNN...")
print("="*60)

results = system.diagnose()

print("\n✓ Inferenza completata!")
print(f"✓ Diagnosi generate per {len(results)} pazienti")


Esecuzione inferenza LNN...

✓ Inferenza completata!
✓ Diagnosi generate per 4 pazienti


## Risultati Diagnosi

Per ogni paziente, il sistema mostra le diagnosi con **bounds di confidenza**.

I bounds `[lower, upper]` rappresentano l'incertezza nella diagnosi:
- **[1.0, 1.0]**: Certezza assoluta
- **[0.7, 0.9]**: Alta probabilità
- **[0.4, 0.6]**: Probabilità media
- **[0.0, 0.3]**: Bassa probabilità

In [35]:
# Stampa risultati
for patient, diagnosis in results.items():
    system.print_diagnosis(patient, diagnosis)


DIAGNOSI PER: ('Sara',)

DIAGNOSI PER: ('Mario',)

DIAGNOSI PER: ('Anna',)

DIAGNOSI PER: ('Luigi',)


## Analisi dei Risultati

Osserva come:

1. **Mario** mostra alta confidenza per Influenza (febbre + dolori)
2. **Anna** mostra alta confidenza per Raffreddore (tosse + mal gola, NO febbre)
3. **Luigi** potrebbe avere Covid E Influenza (sintomi sovrapposti)
4. **Sara** mostra evidenza di Bronchite (tosse + respiro corto)

LNN gestisce automaticamente:
- Incertezza nei sintomi
- Sovrapposizione di diagnosi
- Contraddizioni logiche

## Conclusioni

### Caratteristiche chiave di LNN dimostrate:

1. **Bounds [lower, upper]** rappresentano incertezza
2. **LNN gestisce automaticamente** contraddizioni e incertezze
3. **Inferenza bidirezionale** (modus ponens + tollens)
4. **Interpretabilità completa** - ogni inferenza è tracciabile alle regole

### Vantaggi rispetto ad approcci tradizionali:

| Caratteristica | LNN | Neural Networks | Prolog |
|----------------|-----|-----------------|--------|
| Interpretabilità | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| Gestione incertezza | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| Apprendimento | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐ |
| Ragionamento logico | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ |

### Prossimi Passi

Esplora gli altri esempi del repository:
- Sistema di raccomandazione film
- Sistema ibrido LNN + LLM
- Learning di pesi logici da dati

---

**Nota sull'architettura**: Hai notato che abbiamo importato `MedicalDiagnosisSystem` dal file `.py`? Questo segue il principio **DRY (Don't Repeat Yourself)** - il codice esiste in un solo posto, rendendo la manutenzione più semplice.